# Training Pipeline - ChatKasir

- Nama: Achmad Rif'an (AI-1 Model Architect)
- Minggu: 2 - Pengembangan Fitur Inti (27 April - 1 Mei)


## 1. Setup Google Colab dengan GPU

In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd

# Verifikasi versi
print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")

# Verifikasi GPU
gpus = tf.config.list_physical_devices('GPU')
print(f"\nGPU tersedia: {len(gpus) > 0}")

if gpus:
    # Tampilkan detail GPU yang aktif agar terdokumentasi
    for gpu in gpus:
        print(f"Nama GPU    : {gpu.name}")

    # Aktifkan memory growth - GPU tidak langsung mengambil semua VRAM
    # tapi mengalokasikan secara bertahap sesuai kebutuhan
    # Ini mencegah crash "out of memory" di awal training
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("Memory growth: aktif")
else:
    print("PERINGATAN: GPU tidak aktif! Cek Runtime -> Change runtime type")

TensorFlow : 2.20.0
NumPy      : 2.0.2
Pandas     : 2.2.2

GPU tersedia: False
PERINGATAN: GPU tidak aktif! Cek Runtime -> Change runtime type


## 2. Load Dataset

In [13]:
# load dataset food
url_food_utama = "https://drive.google.com/uc?id=1xpoFjqAT9K0uwzSpVADm_EfKqG7dxVUI"

df_food = pd.read_csv(url_food_utama)
df_food.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18558 entries, 0 to 18557
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   name    18558 non-null  object
dtypes: object(1)
memory usage: 145.1+ KB


In [14]:
# load dataset slang
url_slang_utama = "https://drive.google.com/uc?id=1G14C1qcqOp06Xs1HFiorE3Us_LLtaBs7"

df_slang = pd.read_csv(url_slang_utama)
df_slang.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1231 entries, 0 to 1230
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   slang   1231 non-null   object
 1   formal  1231 non-null   object
dtypes: object(2)
memory usage: 19.4+ KB


In [15]:
# load dataset sintetis
url_synthetic_10000 = "https://drive.google.com/uc?id=15luIrYGJEZpbH-Wf7foXHXo3LBt0MBqU"

df_synthetic = pd.read_csv(url_synthetic_10000)
df_synthetic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10050 entries, 0 to 10049
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   input_text    10050 non-null  object
 1   product       10050 non-null  object
 2   quantity      10050 non-null  int64 
 3   price_satuan  10050 non-null  int64 
 4   pattern       10050 non-null  int64 
dtypes: int64(3), object(2)
memory usage: 392.7+ KB


## 3. Pipeline Data Loading
Ada 4 tahap yang dilakukan:

1. Memuat dan memvalidasi dataset
2. Tokenisasi: mengubah teks percakapan menjadi array integer
3. Encoding label: setiap nama produk dipetakan ke integer
4. Membangun tf.data.Dataset untuk mengirimkan data ke model saat training


In [7]:
# Tahap 1: validasi dataset sintetis

print(f"Shape dataset   : {df_synthetic.shape}")
print(f"Kolom yang ada  : {list(df_synthetic.columns)}")


Shape dataset   : (10050, 5)
Kolom yang ada  : ['input_text', 'product', 'quantity', 'price_satuan', 'pattern']


In [17]:
# validasi nama kolom sesuai kesepakatan
kolom_wajib = ["input_text", "product", "quantity", "price_satuan", "pattern"]

kolom_hilang = [k for k in kolom_wajib if k not in df_synthetic.columns]
if kolom_hilang:
    print(f"ERROR: Kolom berikut tidak ditemukan: {kolom_hilang}")
    print("Minta Faradi (DS-1) untuk menyesuaikan nama kolom!")
else:
    print("Semua kolom sesuai")

Semua kolom sesuai


In [16]:
# tampilkan sampel data untuk inspeksi visual
print(f"Sampel 3 baris pertama:")
print(df_synthetic[["input_text", "product", "quantity", "price_satuan"]].head(3).to_string())

Sampel 3 baris pertama:
                                                                                                                  input_text                        product  quantity  price_satuan
0  minta 7 nasi kari limasari nasi putih ya [SEP] oke kak nasi kari limasari nasi putih harganya 22 ribu totalnya rp 154.000  nasi kari limasari nasi putih         7         22000
1                         bu mau pesen 4 mie goreng djawa [SEP] noted kak mie goreng djawa rp 48.000 per porsi totalnya 192k               mie goreng djawa         4         48000
2                                                                         4 teh obenk dong [SEP] oke kak pesanannya masuk ya                      teh obenk         4            -1


In [25]:
# cek distribusi pattern, memastikan proporsi seimbang
print(f"Distribusi pattern:")

for pattern, count in df_synthetic['pattern'].value_counts().sort_index().items():
    pct = count / len(df_synthetic) * 100
    print(f"Pola {pattern}: {count} baris ({pct:.1f}%)")

Distribusi pattern:
Pola 1: 3922 baris (39.0%)
Pola 2: 3888 baris (38.7%)
Pola 3: 2240 baris (22.3%)


In [26]:
# cek baris dengan harga satuan -1
n_null = (df_synthetic['price_satuan'] == -1).sum()

print(f"Baris tanpa harga satuan (-1): {n_null} ({n_null/len(df_synthetic):.1%})")

Baris tanpa harga satuan (-1): 1750 (17.4%)


In [27]:
# cek apakah ada nilai yang null (NaN)
print(f"Nilai NaN per kolom:")
print(df_synthetic[kolom_wajib].isnull().sum().to_string())

Nilai NaN per kolom:
input_text      0
product         0
quantity        0
price_satuan    0
pattern         0
